# Libraries

In [1]:
import numpy as np
import pandas as pd

from ucimlrepo import fetch_ucirepo
from sklearn.datasets import fetch_california_housing

from tensorflow.keras import layers, models
import tensorflow as tf

from sklearn.metrics import confusion_matrix, f1_score, classification_report, accuracy_score, balanced_accuracy_score, ConfusionMatrixDisplay, roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# own functions
from active_learning import passive_sampling, uncertainty_sampling, label_data, active_learning_bc, active_learning_mc
from active_learning import balanced_weights, tune_params, retrieve_params, cv_classification

# Models

## Breast Cancer

In [2]:
breast_cancer = fetch_ucirepo(id=17)

bcwd_df = pd.concat(
    [breast_cancer.data.features, breast_cancer.data.targets],
    axis=1
)
bcwd_df.head()

,radius1,texture1,perimeter1,area1,smoothness1,compactness1,concavity1,concave_points1,symmetry1,fractal_dimension1,...,texture3,perimeter3,area3,smoothness3,compactness3,concavity3,concave_points3,symmetry3,fractal_dimension3,Diagnosis
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,M
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,M
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,M
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,M
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,M


In [3]:
X = breast_cancer.data.features.to_numpy()
y = (breast_cancer.data.targets).iloc[:, 0].map({"B": 0, "M": 1}).astype("int32").to_numpy()

X_pool, X_test, y_pool, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=51
)

In [4]:
acts = ['relu', 'leaky_relu', 'elu', 'gelu', 'swish']
balance = [False, True]
name = "base"
for state in balance:
    if state:
        name = 'balanced'
    else:
        name = 'base'

    for activation in acts:
        base_params = tune_params(X_pool, y_pool,
                                  loss='binary_crossentropy',
                                  out_units=1, out_act='sigmoid',
                                  k=3, activation=activation,
                                  balanced=False,
                                  random_state=42,
                                  n_trials=80,
                                  export=True, file = f'params/breast_cancer/{name}_{activation}.json'
        )

[I 2026-09-23 22:03:29,931] A new study created in memory with name: no-name-24dba71f-c3cb-43ce-a0ff-95f374362647


[I 2026-09-23 22:03:58,339] Trial 0 finished with value: 0.09482133388519287 and parameters: {'hidden_layer': 66, 'learning_rate': 0.03443006339334324, 'momentum': 0.7239634414397639, 'weight_decay': 0.0007707799579098195}. Best is trial 0 with value: 0.09482133388519287.
[I 2026-09-23 22:04:42,925] Trial 1 finished with value: 0.353429913520813 and parameters: {'hidden_layer': 86, 'learning_rate': 0.0005070587436265495, 'momentum': 0.36121130642843274, 'weight_decay': 0.0008694008697084097}. Best is trial 0 with value: 0.09482133388519287.
[I 2026-09-23 22:05:26,630] Trial 2 finished with value: 0.3215976655483246 and parameters: {'hidden_layer': 60, 'learning_rate': 0.00044191997669171897, 'momentum': 0.515911551210062, 'weight_decay': 0.0006140902224770801}. Best is trial 0 with value: 0.09482133388519287.
[I 2026-09-23 22:06:12,130] Trial 3 finished with value: 0.20186512172222137 and parameters: {'hidden_layer': 92, 'learning_rate': 0.0020764228915147284, 'momentum': 0.21999506357

{'hidden_layer': 96, 'learning_rate': 0.032373116376317075, 'momentum': 0.7809132234484888, 'weight_decay': 1.4346698094774194e-06}
0.08398696035146713


[I 2026-09-23 22:31:13,979] Trial 0 finished with value: 0.10224166512489319 and parameters: {'hidden_layer': 79, 'learning_rate': 0.052754767963352156, 'momentum': 0.17908523735551288, 'weight_decay': 0.00017906767285558243, 'leaky_relu_alpha': 0.01590230789148455}. Best is trial 0 with value: 0.10224166512489319.
[I 2026-09-23 22:31:59,016] Trial 1 finished with value: 0.1346011459827423 and parameters: {'hidden_layer': 21, 'learning_rate': 0.006720837939770634, 'momentum': 0.3357841390733627, 'weight_decay': 6.320713263358466e-05, 'leaky_relu_alpha': 0.011051575355277898}. Best is trial 0 with value: 0.10224166512489319.
[I 2026-09-23 22:32:42,148] Trial 2 finished with value: 0.11142945289611816 and parameters: {'hidden_layer': 27, 'learning_rate': 0.022494270119690754, 'momentum': 0.41450050635359875, 'weight_decay': 0.007746706829439509, 'leaky_relu_alpha': 0.001567975845026253}. Best is trial 0 with value: 0.10224166512489319.
[I 2026-09-23 22:33:25,433] Trial 3 finished with va

{'hidden_layer': 53, 'learning_rate': 0.05819436643372104, 'momentum': 0.7603317133239147, 'weight_decay': 3.2648678371741576e-05, 'leaky_relu_alpha': 0.12373760796970416}
0.08303141593933105


...

{'hidden_layer': 94, 'learning_rate': 0.06622278642632293, 'momentum': 0.7208253099396412, 'weight_decay': 2.618721718157445e-05, 'elu_alpha': 0.36359999453362535}
0.08013296872377396


[I 2026-09-24 01:32:36,998] Trial 0 finished with value: 0.09930193424224854 and parameters: {'hidden_layer': 78, 'learning_rate': 0.004067919586395305, 'momentum': 0.9114221390770122, 'weight_decay': 0.000176375082788953}. Best is trial 0 with value: 0.09930193424224854.
[I 2026-09-24 01:33:20,088] Trial 1 finished with value: 0.18363851308822632 and parameters: {'hidden_layer': 19, 'learning_rate': 0.0020548499190597555, 'momentum': 0.5544956684903524, 'weight_decay': 2.3480733385293147e-05}. Best is trial 0 with value: 0.09930193424224854.
[I 2026-09-24 01:34:04,068] Trial 2 finished with value: 0.11935538798570633 and parameters: {'hidden_layer': 56, 'learning_rate': 0.005328192677086844, 'momentum': 0.5794305072752525, 'weight_decay': 0.0001677576227574097}. Best is trial 0 with value: 0.09930193424224854.
[I 2026-09-24 01:34:47,430] Trial 3 finished with value: 0.12406409531831741 and parameters: {'hidden_layer': 34, 'learning_rate': 0.002472504842041179, 'momentum': 0.8263310125

{'hidden_layer': 72, 'learning_rate': 0.04532427609220552, 'momentum': 0.8632451289416091, 'weight_decay': 0.003071764563711919}
0.08347045630216599


[I 2026-09-24 02:00:40,607] Trial 0 finished with value: 0.26065942645072937 and parameters: {'hidden_layer': 68, 'learning_rate': 0.000362631377812991, 'momentum': 0.7553253189771545, 'weight_decay': 6.522504119075133e-05}. Best is trial 0 with value: 0.26065942645072937.
[I 2026-09-24 02:01:25,761] Trial 1 finished with value: 0.210139200091362 and parameters: {'hidden_layer': 11, 'learning_rate': 0.0006641918231801516, 'momentum': 0.8662309254933128, 'weight_decay': 7.406881922559667e-05}. Best is trial 1 with value: 0.210139200091362.
[I 2026-09-24 02:02:10,717] Trial 2 finished with value: 0.11104544252157211 and parameters: {'hidden_layer': 23, 'learning_rate': 0.009976479865227471, 'momentum': 0.4792395964277038, 'weight_decay': 5.01920849017281e-06}. Best is trial 2 with value: 0.11104544252157211.
[I 2026-09-24 02:02:56,122] Trial 3 finished with value: 0.13400201499462128 and parameters: {'hidden_layer': 45, 'learning_rate': 0.007218203392725648, 'momentum': 0.292199230504373

{'hidden_layer': 55, 'learning_rate': 0.07570302734957246, 'momentum': 0.6202457270290017, 'weight_decay': 9.144592300050395e-06}
0.07814949005842209


In [4]:
acts = ['relu', 'leaky_relu', 'elu', 'gelu', 'swish']
balance = [False, True]

results = []
name = "base"
for state in balance:
    if state:
        name = 'balanced'
    else:
        name = 'base'

    for activation in acts:
        setup = {'activation': activation, 'balanced': state}
        alpha = 0
        params = retrieve_params(f'params/breast_cancer/{name}_{activation}.json')
        if activation == 'leaky_relu':
            alpha = params["leaky_relu_alpha"]
        elif activation == 'elu':
            alpha = params["elu_alpha"]
        result = cv_classification(X_pool, y_pool, out_units=1, out_act='sigmoid', loss="binary_crossentropy", k=3, random_state=42, 
                  hidden_units=params['hidden_layer'], activation=activation, alpha=alpha, 
                  learning_rate=params['learning_rate'], momentum=params['momentum'], weight_decay=params['weight_decay'], balanced=state)
        for key, val in result.items():
            setup[key] = val
    
        results.append(setup)

results_df = pd.DataFrame(results)
results_df.round(4).to_csv(f"results/breast_cancer/cv_validation.csv", index=False)

In [ ]:
active_model = tf.keras.Sequential([
    layers.Input(shape=(X_pool.shape[1],)),
    layers.Dense(32, activation="relu"),
    layers.Dense(1, activation='sigmoid')
])

active_model.compile(
    optimizer=tf.keras.optimizers.SGD(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.AUC(name="auc"), tf.keras.metrics.F1Score(average='macro')]
)

history = active_learning(
    active_model,
    X_pool,
    y_pool,
    X_test,
    y_test,
    passive_sampling,
    num_iter=3,
    unlab_size=0.9,
    random_state=100,
    verbose=True
)

## Dry beans

In [5]:
dry_bean = fetch_ucirepo(id=602)
bean_df = pd.concat(
    [dry_bean.data.features, dry_bean.data.targets],
    axis=1
)
bean_df.head()

,Area,Perimeter,MajorAxisLength,MinorAxisLength,AspectRatio,Eccentricity,ConvexArea,EquivDiameter,Extent,Solidity,Roundness,Compactness,ShapeFactor1,ShapeFactor2,ShapeFactor3,ShapeFactor4,Class
0,28395,610.291,208.178117,173.888747,1.197191,0.549812,28715,190.141097,0.763923,0.988856,0.958027,0.913358,0.007332,0.003147,0.834222,0.998724,SEKER
1,28734,638.018,200.524796,182.734419,1.097356,0.411785,29172,191.272751,0.783968,0.984986,0.887034,0.953861,0.006979,0.003564,0.909851,0.998430,SEKER
2,29380,624.110,212.826130,175.931143,1.209713,0.562727,29690,193.410904,0.778113,0.989559,0.947849,0.908774,0.007244,0.003048,0.825871,0.999066,SEKER
3,30008,645.884,210.557999,182.516516,1.153638,0.498616,30724,195.467062,0.782681,0.976696,0.903936,0.928329,0.007017,0.003215,0.861794,0.994199,SEKER
4,30140,620.134,201.847882,190.279279,1.060798,0.333680,30417,195.896503,0.773098,0.990893,0.984877,0.970516,0.006697,0.003665,0.941900,0.999166,SEKER


In [6]:
X = dry_bean.data.features.to_numpy()
encoder = LabelEncoder()
y = dry_bean.data.targets.to_numpy().ravel()

y = encoder.fit_transform(y)
bean_map = encoder.classes_
n_classes = len(np.unique(y))

X_pool, X_test, y_pool, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=51
)

In [7]:
acts = ['relu', 'leaky_relu', 'elu', 'gelu', 'swish']
name = "base"
for activation in acts:
    base_params = tune_params(X_pool, y_pool,
                              loss='sparse_categorical_crossentropy',
                              out_units=n_classes, out_act='softmax',
                              k=5, activation=activation,
                              balanced=False,
                              random_state=42,
                              n_trials=80,
                              export=True, file = f'params/dry_bean/{name}_{activation}.json'
    )

[I 2026-09-24 08:58:49,653] A new study created in memory with name: no-name-f7990ee7-4d94-46ba-ad02-462e9c640964
[I 2026-09-24 09:03:42,963] Trial 0 finished with value: 0.2359083592891693 and parameters: {'hidden_layer': 31, 'learning_rate': 0.0010504234588356461, 'momentum': 0.5672917461412101, 'weight_decay': 0.00025515089599309465}. Best is trial 0 with value: 0.2359083592891693.
[I 2026-09-24 09:08:12,182] Trial 1 finished with value: 0.20321372151374817 and parameters: {'hidden_layer': 74, 'learning_rate': 0.006456565189986694, 'momentum': 0.252843329131544, 'weight_decay': 1.3376349549684816e-05}. Best is trial 1 with value: 0.20321372151374817.
[I 2026-09-24 09:12:47,467] Trial 2 finished with value: 0.24281105399131775 and parameters: {'hidden_layer': 26, 'learning_rate': 0.0010622162477497206, 'momentum': 0.514092887324837, 'weight_decay': 4.989903460203718e-05}. Best is trial 1 with value: 0.20321372151374817.
[I 2026-09-24 09:14:24,182] Trial 3 finished with value: 0.20411

{'hidden_layer': 107, 'learning_rate': 0.007758378811390079, 'momentum': 0.9414639060544645, 'weight_decay': 0.0001097303171328828}
0.19380834698677063


[I 2026-09-24 10:34:40,683] Trial 0 finished with value: 0.603901743888855 and parameters: {'hidden_layer': 38, 'learning_rate': 0.0002673158232409953, 'momentum': 0.032274798726985035, 'weight_decay': 4.936605963515345e-05, 'leaky_relu_alpha': 0.003109727808447892}. Best is trial 0 with value: 0.603901743888855.
[I 2026-09-24 10:39:23,412] Trial 1 finished with value: 0.23986172676086426 and parameters: {'hidden_layer': 83, 'learning_rate': 0.0012903958116131534, 'momentum': 0.3565891977479561, 'weight_decay': 5.9950085693023085e-05, 'leaky_relu_alpha': 0.008350776258629306}. Best is trial 1 with value: 0.23986172676086426.
[I 2026-09-24 10:44:11,391] Trial 2 finished with value: 0.41579437255859375 and parameters: {'hidden_layer': 78, 'learning_rate': 0.0003941285306908751, 'momentum': 0.20072690389699424, 'weight_decay': 0.001403334804497561, 'leaky_relu_alpha': 0.004809521334064588}. Best is trial 1 with value: 0.23986172676086426.
[I 2026-09-24 10:49:39,590] Trial 3 finished with 

{'hidden_layer': 126, 'learning_rate': 0.038499144390277434, 'momentum': 0.8888210220929265, 'weight_decay': 5.114730565238182e-06, 'leaky_relu_alpha': 0.020452474706357127}
0.19284117221832275


[I 2026-09-24 12:28:10,555] Trial 0 finished with value: 0.20084130764007568 and parameters: {'hidden_layer': 58, 'learning_rate': 0.0094677616697529, 'momentum': 0.9291702662166553, 'weight_decay': 6.679133460540766e-05, 'elu_alpha': 1.2326761011141363}. Best is trial 0 with value: 0.20084130764007568.
[I 2026-09-24 12:29:37,334] Trial 1 finished with value: 0.19823968410491943 and parameters: {'hidden_layer': 70, 'learning_rate': 0.06869919389678714, 'momentum': 0.7515212120359361, 'weight_decay': 6.7135691451640234e-06, 'elu_alpha': 0.10652297231932507}. Best is trial 1 with value: 0.19823968410491943.
[I 2026-09-24 12:33:20,755] Trial 2 finished with value: 0.20361976325511932 and parameters: {'hidden_layer': 40, 'learning_rate': 0.005478168211603529, 'momentum': 0.7312758087345064, 'weight_decay': 4.161424936052871e-06, 'elu_alpha': 1.1496989473310768}. Best is trial 1 with value: 0.19823968410491943.
[I 2026-09-24 12:39:18,291] Trial 3 finished with value: 0.24139642715454102 and

...

{'hidden_layer': 86, 'learning_rate': 0.06916135455563951, 'momentum': 0.8404367717780885, 'weight_decay': 0.0005393788572166543}
0.19460535049438477


[I 2026-09-24 15:43:04,131] Trial 0 finished with value: 0.21250960230827332 and parameters: {'hidden_layer': 46, 'learning_rate': 0.0061649117765305295, 'momentum': 0.026327865999117597, 'weight_decay': 3.4252575969670253e-06}. Best is trial 0 with value: 0.21250960230827332.
[I 2026-09-24 15:49:11,801] Trial 1 finished with value: 0.5587712526321411 and parameters: {'hidden_layer': 94, 'learning_rate': 0.00014306667156999147, 'momentum': 0.5856017619536737, 'weight_decay': 6.598749807683256e-05}. Best is trial 0 with value: 0.21250960230827332.
[I 2026-09-24 15:51:33,281] Trial 2 finished with value: 0.19777460396289825 and parameters: {'hidden_layer': 86, 'learning_rate': 0.01000314757145471, 'momentum': 0.915372569690932, 'weight_decay': 6.87168975312532e-05}. Best is trial 2 with value: 0.19777460396289825.
[I 2026-09-24 15:56:42,992] Trial 3 finished with value: 0.22301384806632996 and parameters: {'hidden_layer': 33, 'learning_rate': 0.006216506625991501, 'momentum': 0.498443313

{'hidden_layer': 57, 'learning_rate': 0.026731908629622256, 'momentum': 0.7455897737662243, 'weight_decay': 3.99937172643707e-06}
0.1956712007522583


[I 2026-09-24 17:48:15,933] Trial 0 finished with value: 0.374485582113266 and parameters: {'hidden_layer': 70, 'learning_rate': 0.00024927065531605896, 'momentum': 0.6492330335921463, 'weight_decay': 1.2087112005111532e-06}. Best is trial 0 with value: 0.374485582113266.
[I 2026-09-24 17:53:57,007] Trial 1 finished with value: 0.3847348690032959 and parameters: {'hidden_layer': 93, 'learning_rate': 0.000497296738567034, 'momentum': 0.24224269611933166, 'weight_decay': 0.00013289185475650556}. Best is trial 0 with value: 0.374485582113266.
[I 2026-09-24 17:56:44,662] Trial 2 finished with value: 0.21049609780311584 and parameters: {'hidden_layer': 17, 'learning_rate': 0.013905546486568707, 'momentum': 0.5172550951043268, 'weight_decay': 2.8984542946793415e-06}. Best is trial 2 with value: 0.21049609780311584.
[I 2026-09-24 18:00:36,074] Trial 3 finished with value: 0.2036658525466919 and parameters: {'hidden_layer': 111, 'learning_rate': 0.003340192661172581, 'momentum': 0.823304634779

KeyboardInterrupt: 

In [9]:
acts = ['relu', 'leaky_relu', 'elu', 'gelu', 'swish']

results = []
name = "base"

for activation in acts:
    setup = {'activation': activation, 'balanced': state}
    alpha = 0
    params = retrieve_params(f'params/dry_bean/{name}_{activation}.json')
    if activation == 'leaky_relu':
        alpha = params["leaky_relu_alpha"]
    elif activation == 'elu':
        alpha = params["elu_alpha"]
    result = cv_classification(X_pool, y_pool, 
               out_units=n_classes, out_act='softmax', loss='sparse_categorical_crossentropy', k=5, random_state=42, 
               hidden_units=params['hidden_layer'], activation=activation, alpha=alpha, 
               learning_rate=params['learning_rate'], momentum=params['momentum'], weight_decay=params['weight_decay'], balanced=False)
    for key, val in result.items():
        setup[key] = val

    results.append(setup)

results_df = pd.DataFrame(results)
results_df.round(4).to_csv(f"results/dry_bean/cv_validation.csv", index=False)

## Student dropout

In [10]:
student_dropout = fetch_ucirepo(id=697)
dropout_df = pd.concat(
    [student_dropout.data.features, student_dropout.data.targets],
    axis=1
)
dropout_df.head()

,Marital Status,Application mode,Application order,Course,Daytime/evening attendance,Previous qualification,Previous qualification (grade),Nacionality,Mother's qualification,Father's qualification,...,Curricular units 2nd sem (credited),Curricular units 2nd sem (enrolled),Curricular units 2nd sem (evaluations),Curricular units 2nd sem (approved),Curricular units 2nd sem (grade),Curricular units 2nd sem (without evaluations),Unemployment rate,Inflation rate,GDP,Target
0,1,17,5,171,1,1,122.0,1,19,12,...,0,0,0,0,0.000000,0,10.8,1.4,1.74,Dropout
1,1,15,1,9254,1,1,160.0,1,1,3,...,0,6,6,6,13.666667,0,13.9,-0.3,0.79,Graduate
2,1,1,5,9070,1,1,122.0,1,37,37,...,0,6,0,0,0.000000,0,10.8,1.4,1.74,Dropout
3,1,17,2,9773,1,1,122.0,1,38,37,...,0,6,10,5,12.400000,0,9.4,-0.8,-3.12,Graduate
4,2,39,1,8014,0,1,100.0,1,37,38,...,0,6,6,6,13.000000,0,13.9,-0.3,0.79,Graduate


In [11]:
X = student_dropout.data.features.to_numpy()
student_encoder = LabelEncoder()
y = student_dropout.data.targets.to_numpy().ravel()

y = student_encoder.fit_transform(y)
dropout_map = student_encoder.classes_
n_classes = len(np.unique(y))

X_pool, X_test, y_pool, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=51
)

In [ ]:
acts = ['relu', 'leaky_relu', 'elu', 'gelu', 'swish']
name = 'base'
for activation in acts:
    base_params = tune_params(X_pool, y_pool,
                              loss='sparse_categorical_crossentropy',
                              out_units=n_classes, out_act='softmax',
                              k=5, activation=activation,
                              balanced=False,
                              random_state=42,
                              n_trials=80,
                              export=True, file = f'params/student_dropout/{name}_{activation}.json'
    )

[I 2026-09-24 20:18:40,418] A new study created in memory with name: no-name-e216e3c9-b63d-426e-a7af-a7a71cc14175
[I 2026-09-24 20:21:20,339] Trial 0 finished with value: 0.685571551322937 and parameters: {'hidden_layer': 93, 'learning_rate': 0.000332670473722622, 'momentum': 0.43414183367375986, 'weight_decay': 1.815090504102778e-05}. Best is trial 0 with value: 0.685571551322937.
[I 2026-09-24 20:23:54,482] Trial 1 finished with value: 0.7891916036605835 and parameters: {'hidden_layer': 53, 'learning_rate': 0.00023365750923479533, 'momentum': 0.1581928059374333, 'weight_decay': 0.001285086752112229}. Best is trial 0 with value: 0.685571551322937.
[I 2026-09-24 20:26:25,343] Trial 2 finished with value: 0.7188781499862671 and parameters: {'hidden_layer': 21, 'learning_rate': 0.0003544044690087883, 'momentum': 0.5258776857070462, 'weight_decay': 1.9429070217827516e-06}. Best is trial 0 with value: 0.685571551322937.
[I 2026-09-24 20:27:18,660] Trial 3 finished with value: 0.58889377117

{'hidden_layer': 116, 'learning_rate': 0.07891131882926719, 'momentum': 0.24574476009598603, 'weight_decay': 0.007319333737600483}
0.575970470905304


[I 2026-09-24 21:27:12,630] Trial 0 finished with value: 0.6275041699409485 and parameters: {'hidden_layer': 55, 'learning_rate': 0.001301097691867977, 'momentum': 0.14641735980957374, 'weight_decay': 1.2867470952177964e-05, 'leaky_relu_alpha': 0.001565679063972322}. Best is trial 0 with value: 0.6275041699409485.
[I 2026-09-24 21:29:47,534] Trial 1 finished with value: 0.7303287386894226 and parameters: {'hidden_layer': 97, 'learning_rate': 0.00025297626650023873, 'momentum': 0.18665515641500569, 'weight_decay': 3.0607843654848856e-06, 'leaky_relu_alpha': 0.06634204665438757}. Best is trial 0 with value: 0.6275041699409485.
[I 2026-09-24 21:32:26,608] Trial 2 finished with value: 0.6388384699821472 and parameters: {'hidden_layer': 12, 'learning_rate': 0.001921819171820562, 'momentum': 0.10383898604094759, 'weight_decay': 0.00012192339366717742, 'leaky_relu_alpha': 0.0889338021451944}. Best is trial 0 with value: 0.6275041699409485.
[I 2026-09-24 21:35:23,848] Trial 3 finished with val

In [ ]:
acts = ['relu', 'leaky_relu', 'elu', 'gelu', 'swish']
name = "base"
for activation in acts:
    setup = {'activation': activation, 'balanced': state}
    alpha = 0
    params = retrieve_params(f'params/student_dropout/{name}_{activation}.json')
    if activation == 'leaky_relu':
        alpha = params["leaky_relu_alpha"]
    elif activation == 'elu':
        alpha = params["elu_alpha"]
    result = cv_classification(X_pool, y_pool, out_units=n_classes, out_act='softmax', 
                    loss='sparse_categorical_crossentropy', k=3, random_state=42, 
                    hidden_units=params['hidden_layer'], activation=activation, alpha=alpha, 
                    learning_rate=params['learning_rate'], momentum=params['momentum'], weight_decay=params['weight_decay'], balanced=False)
    for key, val in result.items():
        setup[key] = val

    results.append(setup)

results_df = pd.DataFrame(results)
results_df.round(4).to_csv(f"results/student_dropout/cv_validation.csv", index=False)